# Ground Truth Analysis

This notebook evaluates the considered partitioning algorithms on synthetic Gaussian partition graphs with a known ground-truth community structure.

The following algorithms are evaluated:

- `leiden`
- `leiden_mdgp`
- `kapoce`
- `mdgp_plateau`

The analysis focuses on two aspects:

1. **MDGP solution quality:** comparison of the partition density attained by an algorithm with the density of the ground-truth partition.
2. **Ground-truth reconstruction:** comparison of the resulting partition with the planted community structure using the clustering F-score.

Results are aggregated by graph size, density regime, noise level, community-size class, and algorithm.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

## Configuration

In [2]:
SIZE_ORDER = ["small", "large"]
REGIME_ORDER = ["sparse", "dense"]
NOISE_ORDER = [0.05, 0.20]
COMMUNITY_SIZE_ORDER = ["tiny", "small", "large",]
ALGORITHM_ORDER = ["leiden", "leiden_mdgp", "kapoce", "mdgp_plateau"]

DATA_ROOT = Path("../../data/ground_truth")

RESULTS_DIR = Path("../../results/experiment4")
RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load and verify experiment data

Before the analysis, the notebook verifies that all expected algorithms and dataset groups are contained in the result file.

For every graph instance, exactly one result per algorithm is expected.

In [3]:
raw = pd.read_csv(RAW_RESULTS_FILE)

required_columns = {
    "dataset",
    "size_class",
    "regime",
    "community_size_class",
    "noise",
    "p_in",
    "p_out",
    "instance",
    "n",
    "m",
    "edge_density",
    "ground_truth_density",
    "ground_truth_num_clusters",
    "ground_truth_max_cluster_size",
    "ground_truth_avg_cluster_size",
    "ground_truth_partition",
    "algorithm",
    "density",
    "num_clusters",
    "max_cluster_size",
    "avg_cluster_size",
    "runtime",
    "partition",
}

missing_columns = required_columns.difference(raw.columns)

if missing_columns:
    raise ValueError("Missing required columns: "  + ", ".join(sorted(missing_columns)))

print(f"Loaded {len(raw):,} rows from {RAW_RESULTS_FILE}")

Loaded 20,000 rows from ../../results/experiment4/raw_results.csv


In [4]:
raw["noise"] = raw["noise"].astype(float)

raw["size_class"] = pd.Categorical(
    raw["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

raw["regime"] = pd.Categorical(
    raw["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)

raw["community_size_class"] = pd.Categorical(
    raw["community_size_class"],
    categories=COMMUNITY_SIZE_ORDER,
    ordered=True,
)

raw["algorithm"] = pd.Categorical(
    raw["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

GROUP_COLUMNS = ["size_class", "regime", "noise", "community_size_class"]

experiment_check = (
    raw
    .groupby(GROUP_COLUMNS + ["algorithm"], observed=True, as_index=False)
    .agg(
        num_instances=("instance", "nunique"),
        num_rows=("instance", "size"),
    )
    .sort_values(GROUP_COLUMNS + ["algorithm"])
    .reset_index(drop=True)
)

experiment_check

,size_class,regime,noise,community_size_class,algorithm,num_instances,num_rows
0,small,sparse,0.05,small,leiden,250,250
1,small,sparse,0.05,small,leiden_mdgp,250,250
2,small,sparse,0.05,small,kapoce,250,250
3,small,sparse,0.05,small,mdgp_plateau,250,250
4,small,sparse,0.05,large,leiden,250,250
...,...,...,...,...,...,...,...
75,large,dense,0.20,small,mdgp_plateau,250,250
76,large,dense,0.20,large,leiden,250,250
77,large,dense,0.20,large,leiden_mdgp,250,250
78,large,dense,0.20,large,kapoce,250,250


## Generated instance characteristics

Before comparing the algorithms, the generated graph instances are summarized.

Each graph occurs once in this table, independent of the number of evaluated algorithms. The summary is used to verify the graph sizes, edge densities, and ground-truth community structures generated for the different noise levels and community-size classes.

In [5]:
def minimum_ground_truth_cluster_size(partition_value: str) -> int:
    partition = json.loads(partition_value)

    return min(len(cluster) for cluster in partition)

In [6]:
instance_columns = [
    "size_class",
    "regime",
    "noise",
    "community_size_class",
    "instance",
    "n",
    "m",
    "edge_density",
    "ground_truth_density",
    "ground_truth_num_clusters",
    "ground_truth_max_cluster_size",
    "ground_truth_avg_cluster_size",
    "ground_truth_partition",
]

instances = (
    raw[instance_columns]
    .drop_duplicates(subset=["instance"])
    .copy()
)

instances["average_degree"] = 2 * instances["m"] / instances["n"]

instances["ground_truth_min_cluster_size"] = instances["ground_truth_partition"].apply(minimum_ground_truth_cluster_size)

instance_summary = (
    instances
    .groupby(GROUP_COLUMNS, observed=True, as_index=False)
    .agg(
        num_instances=("instance", "nunique"),
        mean_n=("n", "mean"),
        mean_average_degree=("average_degree", "mean"),
        mean_edge_density=("edge_density", "mean"),
        mean_ground_truth_density=("ground_truth_density", "mean"),
        mean_ground_truth_num_clusters=("ground_truth_num_clusters", "mean"),
        mean_ground_truth_avg_cluster_size=("ground_truth_avg_cluster_size", "mean"),
        mean_ground_truth_max_cluster_size=("ground_truth_max_cluster_size", "mean"),
    )
    .sort_values(GROUP_COLUMNS)
    .reset_index(drop=True)
)

instance_summary

,size_class,regime,noise,community_size_class,num_instances,mean_n,mean_average_degree,mean_edge_density,mean_ground_truth_density,mean_ground_truth_num_clusters,mean_ground_truth_avg_cluster_size,mean_ground_truth_max_cluster_size
0,small,sparse,0.05,small,250,210.776,9.534450,0.045889,37.314873,9.844,21.748047,37.300
1,small,sparse,0.05,large,250,210.776,9.469516,0.045580,18.881599,4.948,44.037852,66.736
2,small,sparse,0.20,small,250,210.776,9.357357,0.045006,31.287733,9.936,21.642124,37.688
3,small,sparse,0.20,large,250,210.776,9.160927,0.044174,15.857297,5.220,41.859152,67.364
4,small,dense,0.05,small,250,210.776,19.005303,0.091539,74.189401,10.148,21.114860,37.740
5,small,dense,0.05,large,250,210.776,18.033941,0.086948,37.429620,5.456,39.980383,64.680
6,small,dense,0.20,small,250,210.776,18.383107,0.088537,62.339932,10.428,20.724858,37.852
7,small,dense,0.20,large,250,210.776,17.744692,0.085428,31.513902,5.552,39.291457,65.580
8,large,sparse,0.05,tiny,250,913.656,9.788976,0.011469,278.742269,71.568,12.785026,27.076
9,large,sparse,0.05,small,250,1013.204,10.285781,0.010975,37.968460,8.780,118.053352,187.732


In [7]:
def target_average_degree(n: int, regime: str) -> float:
    if regime == "sparse":
        return 8.0

    if regime == "dense":
        return max(16.0, 0.04 * n)


instances["target_average_degree"] = instances.apply(
    lambda row: target_average_degree(n=row["n"], regime=str(row["regime"])),
    axis=1,
)

In [8]:
instance_files = {path.stem: path for path in DATA_ROOT.rglob("*.json")}

print(f"Found {len(instance_files):,} generated ground-truth instances.")

Found 5,000 generated ground-truth instances.


In [9]:
def external_edge_fraction(edges: list[list[int]], ground_truth: list[list[int]]) -> float:
    cluster_by_node: dict[int, int] = {}

    for cluster_id, cluster in enumerate(ground_truth):
        for node in cluster:
            cluster_by_node[node] = cluster_id

    external_edges = sum(cluster_by_node[u] != cluster_by_node[v] for u, v in edges)

    return external_edges / len(edges)

In [10]:
def missing_internal_edge_fraction(edges: list[list[int]], ground_truth: list[list[int]]) -> float:
    cluster_by_node: dict[int, int] = {}

    for cluster_id, cluster in enumerate(ground_truth):
        for node in cluster:
            cluster_by_node[node] = cluster_id

    internal_edges = sum(cluster_by_node[u] == cluster_by_node[v] for u, v in edges)
    possible_internal_edges = sum(len(cluster) * (len(cluster) - 1) // 2 for cluster in ground_truth)

    if possible_internal_edges == 0:
        return 0.0

    missing_internal_edges = possible_internal_edges - internal_edges

    return missing_internal_edges / possible_internal_edges

In [11]:
instance_metadata_columns = [
    "size_class",
    "regime",
    "community_size_class",
    "noise",
    "instance",
    "p_in",
    "p_out",
]

instance_metadata = (
    raw[instance_metadata_columns]
    .drop_duplicates(subset=["instance"])
    .copy()
)

structure_rows = []

for row in instance_metadata.itertuples(index=False):
    instance_path = instance_files[row.instance]

    with instance_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    structure_rows.append(
        {
            "instance": row.instance,
            "external_edge_fraction": external_edge_fraction(edges=data["edges"], ground_truth=data["ground_truth"]),
            "missing_internal_edge_fraction": missing_internal_edge_fraction(edges=data["edges"], ground_truth=data["ground_truth"]),
        }
    )

structure_data = pd.DataFrame(structure_rows)

instance_metadata = instance_metadata.merge(structure_data, on="instance", validate="one_to_one")

instance_metadata

,size_class,regime,community_size_class,noise,instance,p_in,p_out,external_edge_fraction,missing_internal_edge_fraction
0,large,dense,large,0.05,large_gaussian_partition_dense_communities-lar...,0.190525,0.002504,0.055997,0.812147
1,large,dense,large,0.05,large_gaussian_partition_dense_communities-lar...,0.191493,0.002501,0.037050,0.810037
2,large,dense,large,0.05,large_gaussian_partition_dense_communities-lar...,0.191562,0.002501,0.044957,0.806814
3,large,dense,large,0.05,large_gaussian_partition_dense_communities-lar...,0.190955,0.002503,0.061802,0.810476
4,large,dense,large,0.05,large_gaussian_partition_dense_communities-lar...,0.190151,0.002504,0.036021,0.808015
...,...,...,...,...,...,...,...,...,...
4995,small,sparse,small,0.20,small_gaussian_partition_sparse_communities-sm...,0.284444,0.007637,0.198039,0.712175
4996,small,sparse,small,0.20,small_gaussian_partition_sparse_communities-sm...,0.272340,0.007565,0.144737,0.715792
4997,small,sparse,small,0.20,small_gaussian_partition_sparse_communities-sm...,0.272340,0.007390,0.143172,0.727209
4998,small,sparse,small,0.20,small_gaussian_partition_sparse_communities-sm...,0.297674,0.008020,0.180556,0.714187


In [12]:
degree_summary = (
    instances
    .groupby(["size_class", "regime"], observed=True, as_index=False,)
    .agg(
        mean_target_degree=("target_average_degree", "mean"),
        mean_realized_degree=("average_degree", "mean"),
    )
    .sort_values(["size_class", "regime"])
    .reset_index(drop=True)
)

degree_summary

,size_class,regime,mean_target_degree,mean_realized_degree
0,small,sparse,8.00000,9.380563
1,small,dense,16.00000,18.291761
2,large,sparse,8.00000,9.800944
3,large,dense,40.52816,47.149936


In [13]:
noise_summary = (
    instance_metadata
    .groupby(GROUP_COLUMNS, observed=True, as_index=False)
    .agg(
        mean_p_in=("p_in", "mean"),
        mean_p_out=("p_out", "mean"),
        mean_external_edge_fraction=("external_edge_fraction", "mean"),
        mean_missing_internal_edge_fraction=("missing_internal_edge_fraction", "mean"),
    )
    .sort_values(GROUP_COLUMNS)
    .reset_index(drop=True)
)

noise_summary

,size_class,regime,noise,community_size_class,mean_p_in,mean_p_out,mean_external_edge_fraction,mean_missing_internal_edge_fraction
0,small,sparse,0.05,small,0.373388,0.002135,0.042472,0.625485
1,small,sparse,0.05,large,0.184291,0.002403,0.042457,0.815032
2,small,sparse,0.20,small,0.314432,0.008541,0.170626,0.685524
3,small,sparse,0.20,large,0.155193,0.009613,0.172712,0.844077
4,small,dense,0.05,small,0.746777,0.004271,0.042257,0.252562
5,small,dense,0.05,large,0.368583,0.004806,0.044353,0.631942
6,small,dense,0.20,small,0.628865,0.017082,0.172699,0.370996
7,small,dense,0.20,large,0.310385,0.019225,0.179433,0.689831
8,large,sparse,0.05,tiny,0.660870,0.000476,0.042212,0.338535
9,large,sparse,0.05,small,0.081876,0.000476,0.038306,0.918021


In [14]:
community_size_summary = (
    instances
    .groupby(
        GROUP_COLUMNS,
        observed=True,
        as_index=False,
    )
    .agg(
        num_instances=(
            "instance",
            "nunique",
        ),
        mean_num_clusters=(
            "ground_truth_num_clusters",
            "mean",
        ),
        mean_cluster_size=(
            "ground_truth_avg_cluster_size",
            "mean",
        ),
        mean_min_cluster_size=(
            "ground_truth_min_cluster_size",
            "mean",
        ),
        mean_max_cluster_size=(
            "ground_truth_max_cluster_size",
            "mean",
        ),
    )
    .sort_values(GROUP_COLUMNS)
    .reset_index(drop=True)
)

community_size_summary

,size_class,regime,noise,community_size_class,num_instances,mean_num_clusters,mean_cluster_size,mean_min_cluster_size,mean_max_cluster_size
0,small,sparse,0.05,small,250,9.844,21.748047,7.980,37.300
1,small,sparse,0.05,large,250,4.948,44.037852,22.656,66.736
2,small,sparse,0.20,small,250,9.936,21.642124,6.708,37.688
3,small,sparse,0.20,large,250,5.220,41.859152,18.356,67.364
4,small,dense,0.05,small,250,10.148,21.114860,6.808,37.740
5,small,dense,0.05,large,250,5.456,39.980383,17.032,64.680
6,small,dense,0.20,small,250,10.428,20.724858,5.996,37.852
7,small,dense,0.20,large,250,5.552,39.291457,15.000,65.580
8,large,sparse,0.05,tiny,250,71.568,12.785026,2.120,27.076
9,large,sparse,0.05,small,250,8.780,118.053352,59.084,187.732


## LaTeX table for generated graph structure

The following table summarizes the generation probabilities and the realized fraction of inter-community edges for all ground-truth dataset groups.

## Solution quality relative to the ground truth

For each algorithm and instance, the density ratio is defined as

$
\frac{d(P_{\mathrm{GT}})}{d(P_{\mathrm{algorithm}})}.
$

Interpretation:

- `1.0`: algorithm and ground truth have the same MDGP density,
- less than `1.0`: the algorithm finds a partition with higher MDGP density than the ground truth,
- greater than `1.0`: the algorithm finds a partition with lower MDGP density.

In [15]:
raw["density_ratio_to_ground_truth"] = raw["ground_truth_density"] / raw["density"]

quality_summary = (
    raw
    .groupby(GROUP_COLUMNS +["algorithm"], observed=True, as_index=False,)
    .agg(
        mean_density_ratio_to_ground_truth=("density_ratio_to_ground_truth", "mean"),
        min_density_ratio_to_ground_truth=("density_ratio_to_ground_truth", "min"),
        max_density_ratio_to_ground_truth=("density_ratio_to_ground_truth", "max"),
    )
    .sort_values(GROUP_COLUMNS +["algorithm"])
    .reset_index(drop=True)
)

quality_table = (
    quality_summary
    .pivot(
        index=GROUP_COLUMNS,
        columns="algorithm",
        values="mean_density_ratio_to_ground_truth",
    )
    .reindex(columns=ALGORITHM_ORDER)
    .reset_index()
)

quality_table

algorithm,size_class,regime,noise,community_size_class,leiden,leiden_mdgp,kapoce,mdgp_plateau
0,small,sparse,0.05,small,1.020245,0.593937,0.553467,0.523221
1,small,sparse,0.05,large,1.001671,0.324482,0.291923,0.283674
2,small,sparse,0.20,small,1.057122,0.539093,0.475983,0.456736
3,small,sparse,0.20,large,1.017672,0.287039,0.251741,0.246905
4,small,dense,0.05,small,1.017746,1.005963,0.978956,0.884242
5,small,dense,0.05,large,1.007005,0.575148,0.521548,0.490432
6,small,dense,0.20,small,1.055919,0.988658,0.882436,0.773315
7,small,dense,0.20,large,1.030231,0.532521,0.446406,0.423690
8,large,sparse,0.05,tiny,1.273698,0.917095,0.915176,0.823549
9,large,sparse,0.05,small,1.000723,0.153211,0.134479,0.133613


## Ground-truth reconstruction using the clustering F-score

The second analysis measures how closely the partition produced by an algorithm reconstructs the planted ground-truth community structure.

Let

$
P_{\mathrm{GT}} = \{C_1^*, \dots, C_{k^*}^*\}
$

denote the ground-truth partition and

$
P = \{C_1, \dots, C_k\}
$

the partition produced by an algorithm.

For every pair consisting of a ground-truth cluster $C_i^*$ and an algorithmic cluster $C_j$, precision and recall are defined as

$
P_{ij} = \frac{|C_i^* \cap C_j|}{|C_j|}
$

and

$
R_{ij} = \frac{|C_i^* \cap C_j|}{|C_i^*|}.
$

Their F-score is

$
F_{ij} = \frac{2P_{ij}R_{ij}}{P_{ij}+R_{ij}}.
$

For every ground-truth cluster, only the best matching algorithmic cluster is considered. The overall clustering F-score is therefore

$
F(P, P_{\mathrm{GT}}) = \frac{1}{n} \sum_{i=1}^{k^*} |C_i^*| \max_{1 \leq j \leq k} F_{ij}.
$

The score lies between 0 and 1. A value of 1 indicates an exact reconstruction of the ground-truth partition, while lower values indicate increasing structural differences.

The matching is performed independently for each ground-truth cluster. Thus, the same algorithmic cluster may be the best match for multiple ground-truth clusters.

In [16]:
def parse_partition(value: str) -> list[set[int]]:
    return [set(cluster) for cluster in json.loads(value)]


def cluster_f_score(ground_truth_cluster: set[int], cluster: set[int]) -> float:
    intersection_size = len(ground_truth_cluster & cluster)

    if intersection_size == 0:
        return 0.0

    return 2.0 * intersection_size / (len(ground_truth_cluster) + len(cluster))


def clustering_f_score(ground_truth: list[set[int]], partition: list[set[int]]) -> float:
    n = sum(len(cluster) for cluster in ground_truth)
    weighted_score = 0.0

    for ground_truth_cluster in ground_truth:
        best_f_score = max((cluster_f_score(ground_truth_cluster, cluster) for cluster in partition), default=0.0)
        weighted_score += len(ground_truth_cluster) * best_f_score

    return weighted_score / n


def row_f_score(row: pd.Series) -> float:
    ground_truth = parse_partition(row["ground_truth_partition"])
    partition = parse_partition(row["partition"])

    return clustering_f_score(ground_truth=ground_truth, partition=partition)

In [17]:
raw["f_score"] = raw.apply(row_f_score, axis=1)

f_score_summary = (
    raw
    .groupby(GROUP_COLUMNS + ["algorithm"], observed=True, as_index=False,)
    .agg(
        mean_f_score=("f_score", "mean"),
        min_f_score=("f_score", "min"),
        max_f_score=("f_score", "max"),
    )
    .sort_values(GROUP_COLUMNS + ["algorithm"])
    .reset_index(drop=True)
)

f_score_table = (
    f_score_summary
    .pivot(
        index=GROUP_COLUMNS,
        columns="algorithm",
        values="mean_f_score",
    )
    .reindex(columns=ALGORITHM_ORDER)
    .reset_index()
)

f_score_table

algorithm,size_class,regime,noise,community_size_class,leiden,leiden_mdgp,kapoce,mdgp_plateau
0,small,sparse,0.05,small,0.984203,0.291107,0.437192,0.333056
1,small,sparse,0.05,large,0.994615,0.153488,0.204149,0.172593
2,small,sparse,0.20,small,0.949439,0.260339,0.383836,0.309757
3,small,sparse,0.20,large,0.957628,0.143843,0.190730,0.167793
4,small,dense,0.05,small,0.986080,0.437919,0.959846,0.462973
5,small,dense,0.05,large,0.993776,0.189491,0.341219,0.220513
6,small,dense,0.20,small,0.956866,0.321757,0.816398,0.418860
7,small,dense,0.20,large,0.973837,0.158952,0.289526,0.207589
8,large,sparse,0.05,tiny,0.840299,0.496901,0.858481,0.557075
9,large,sparse,0.05,small,0.998318,0.065576,0.080312,0.072415


## LaTeX helper functions

The following functions format algorithm names and numerical values for the thesis tables.

In [18]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"


def format_percent(value: float, decimals: int) -> str:
    return f"{truncate_number(100 * value, decimals):.{decimals}f}" + r"\,\%"



## Build LaTeX tables

In [42]:
def make_ground_truth_community_size_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    noise_order = [0.05, 0.20]

    community_order_by_size = {
        "small": ["small", "large"],
        "large": ["tiny", "small", "large"],
    }

    lines = [
        r"\begin{table}[p]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Datensatz & $\mu$ "
        r"& \shortstack{Community-\\größe} & \shortstack{Anzahl\\Communitys} & \shortstack{Mittlere\\Größe} & \shortstack{Mittleres\\Minimum} & \shortstack{Mittleres\\Maximum} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(dataset_order):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)].copy()

        community_order = community_order_by_size[size_class]

        dataset_row_count = len(noise_order) * len(community_order)

        current_dataset_row = 0

        for noise_index, noise in enumerate(noise_order):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)].copy()

            for community_index, community_size in enumerate(community_order):
                part = noise_df[noise_df["community_size_class"] == community_size]

                if part.empty:
                    continue

                row = part.iloc[0]

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_dataset_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {row['mean_num_clusters']:.1f} "
                    f"& {row['mean_cluster_size']:.1f} "
                    f"& {row['mean_min_cluster_size']:.1f} "
                    f"& {row['mean_max_cluster_size']:.1f} "
                    r"\\"
                )

                current_dataset_row += 1

            if noise_index < len(noise_order) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(dataset_order) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [43]:
ground_truth_community_size_latex = (
    make_ground_truth_community_size_latex_table(
        community_size_summary,
        caption=(
            "Mittlere Anzahl und Größe der Ground-Truth-Communitys. Minimum und Maximum bezeichnen jeweils den Mittelwert der kleinsten beziehungsweise größten Community einer Instanz."
        ),
        label="tab:ground_truth_community_sizes",
    )
)

print(ground_truth_community_size_latex)

\begin{table}[p]
\centering
\caption{Mittlere Anzahl und Größe der Ground-Truth-Communitys. Minimum und Maximum bezeichnen jeweils den Mittelwert der kleinsten beziehungsweise größten Community einer Instanz.}
\label{tab:ground_truth_community_sizes}
\small
\begin{tabular}{p{2.0cm}rlrrrr}
\toprule
Datensatz & $\mu$ & \shortstack{Community-\\größe} & \shortstack{Anzahl\\Communitys} & \shortstack{Mittlere\\Größe} & \shortstack{Mittleres\\Minimum} & \shortstack{Mittleres\\Maximum} \\
\midrule
\multirow{4}{*}{small sparse} & \multirow{2}{*}{0.05} & small & 9.8 & 21.7 & 8.0 & 37.3 \\
 &  & large & 4.9 & 44.0 & 22.7 & 66.7 \\
\cmidrule(l){2-7}
 & \multirow{2}{*}{0.20} & small & 9.9 & 21.6 & 6.7 & 37.7 \\
 &  & large & 5.2 & 41.9 & 18.4 & 67.4 \\
\midrule
\multirow{4}{*}{small dense} & \multirow{2}{*}{0.05} & small & 10.1 & 21.1 & 6.8 & 37.7 \\
 &  & large & 5.5 & 40.0 & 17.0 & 64.7 \\
\cmidrule(l){2-7}
 & \multirow{2}{*}{0.20} & small & 10.4 & 20.7 & 6.0 & 37.9 \\
 &  & large & 5.6 & 39.3 & 

In [23]:
def make_ground_truth_degree_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    size_order = ["small", "large"]
    regime_order = ["sparse", "dense"]

    lines = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lrr}",
        r"\toprule",
        r"Datensatz & \shortstack{Mittlerer\\Zielgrad} & \shortstack{Mittlerer realisierter\\Knotengrad} \\",
        r"\midrule",
    ]

    for size_class in size_order:
        for regime in regime_order:
            part = df[(df["size_class"] == size_class) & (df["regime"] == regime)]

            row = part.iloc[0]

            lines.append(
                f"{size_class} {regime} & {row['mean_target_degree']:.1f} & {row['mean_realized_degree']:.1f} "
                r"\\"
            )

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [24]:
ground_truth_degree_latex = make_ground_truth_degree_latex_table(
    degree_summary,
    caption=("Mittlerer Zielgrad und mittlerer realisierter durchschnittlicher Knotengrad der Ground-Truth-Instanzen."),
    label="tab:ground_truth_density",
)

print(ground_truth_degree_latex)

\begin{table}[H]
\centering
\caption{Mittlerer Zielgrad und mittlerer realisierter durchschnittlicher Knotengrad der Ground-Truth-Instanzen.}
\label{tab:ground_truth_density}
\begin{tabular}{lrr}
\toprule
Datensatz & \shortstack{Mittlerer\\Zielgrad} & \shortstack{Mittlerer realisierter\\Knotengrad} \\
\midrule
small sparse & 8.0 & 9.4 \\
small dense & 16.0 & 18.3 \\
large sparse & 8.0 & 9.8 \\
large dense & 40.5 & 47.1 \\
\bottomrule
\end{tabular}
\end{table}


In [44]:
def make_ground_truth_noise_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    noise_order = [0.05, 0.20]

    community_order_by_size = {
        "small": ["small", "large"],
        "large": ["tiny", "small", "large"],
    }

    lines = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Datensatz & $\mu$ & \shortstack{Community-\\größe} & $p_{\mathrm{in}}$ & $p_{\mathrm{out}}$ & \shortstack{Anteil externer\\Kanten} & \shortstack{Anteil fehlender\\interner Kanten} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(dataset_order):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)].copy()

        community_order = community_order_by_size[size_class]

        dataset_row_count = len(noise_order) * len(community_order)

        current_dataset_row = 0

        for noise_index, noise in enumerate(noise_order):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)].copy()

            for community_index, community_size in enumerate(community_order):
                part = noise_df[noise_df["community_size_class"] == community_size]

                if part.empty:
                    continue

                row = part.iloc[0]

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_dataset_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {format_percent(row['mean_p_in'], 2)} "
                    f"& {format_percent(row['mean_p_out'], 2)} "
                    f"& {format_percent(row['mean_external_edge_fraction'], 2)} "
                    f"& {format_percent(row['mean_missing_internal_edge_fraction'], 2)} "
                    r"\\"
                )

                current_dataset_row += 1

            if noise_index < len(noise_order) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(dataset_order) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [45]:
ground_truth_noise_latex = make_ground_truth_noise_latex_table(
    noise_summary,
    caption="Mittlere Kantenwahrscheinlichkeiten sowie mittlere realisierte Anteile externer und fehlender interner Kanten der Ground-Truth-Communitys.",
    label="tab:ground_truth_noise",
)

print(ground_truth_noise_latex)

\begin{table}[H]
\centering
\caption{Mittlere Kantenwahrscheinlichkeiten sowie mittlere realisierte Anteile externer und fehlender interner Kanten der Ground-Truth-Communitys.}
\label{tab:ground_truth_noise}
\small
\begin{tabular}{p{2.0cm}rlrrrr}
\toprule
Datensatz & $\mu$ & \shortstack{Community-\\größe} & $p_{\mathrm{in}}$ & $p_{\mathrm{out}}$ & \shortstack{Anteil externer\\Kanten} & \shortstack{Anteil fehlender\\interner Kanten} \\
\midrule
\multirow{4}{*}{small sparse} & \multirow{2}{*}{0.05} & small & 37.33\,\% & 0.21\,\% & 4.24\,\% & 62.54\,\% \\
 &  & large & 18.42\,\% & 0.24\,\% & 4.24\,\% & 81.50\,\% \\
\cmidrule(l){2-7}
 & \multirow{2}{*}{0.20} & small & 31.44\,\% & 0.85\,\% & 17.06\,\% & 68.55\,\% \\
 &  & large & 15.51\,\% & 0.96\,\% & 17.27\,\% & 84.40\,\% \\
\midrule
\multirow{4}{*}{small dense} & \multirow{2}{*}{0.05} & small & 74.67\,\% & 0.42\,\% & 4.22\,\% & 25.25\,\% \\
 &  & large & 36.85\,\% & 0.48\,\% & 4.43\,\% & 63.19\,\% \\
\cmidrule(l){2-7}
 & \multirow{2}{*}{

In [27]:
def make_ground_truth_quality_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    noise_order = [0.05, 0.20]
    community_order_by_size = {
        "small": ["small", "large"],
        "large": ["tiny", "small", "large"],
    }

    algorithm_columns = [
        "leiden",
        "leiden_mdgp",
        "kapoce",
        "mdgp_plateau",
    ]

    lines = [
        r"\begin{table}[p]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Datensatz & $\mu$ & \shortstack{Community-\\größe} & \texttt{Leiden} & \shortstack{\texttt{Leiden-}\\\texttt{MDGP}} & \texttt{KaPoCE} & \shortstack{\textls[-50]{\textsc{MDGP-}}\\\textls[-50]{\textsc{Plateau}}} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(dataset_order):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)].copy()

        community_order = community_order_by_size[size_class]

        dataset_row_count = len(noise_order) * len(community_order)

        current_dataset_row = 0

        for noise_index, noise in enumerate(noise_order):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)].copy()

            for community_index, community_size in enumerate(community_order):
                part = noise_df[noise_df["community_size_class"] == community_size]

                row = part.iloc[0]

                values = {algorithm: row[algorithm] for algorithm in algorithm_columns}
                best_quality = min(values.values())

                formatted_values = {}

                for algorithm, value in values.items():
                    formatted = format_number(value, 4)

                    if np.isclose(value, best_quality):
                        formatted = rf"\textbf{{{formatted}}}"

                    formatted_values[algorithm] = formatted

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_dataset_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {formatted_values['leiden']} "
                    f"& {formatted_values['leiden_mdgp']} "
                    f"& {formatted_values['kapoce']} "
                    f"& {formatted_values['mdgp_plateau']} "
                    r"\\"
                )

                current_dataset_row += 1

            if noise_index < len(noise_order) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(dataset_order) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [33]:
ground_truth_quality_latex = make_ground_truth_quality_latex_table(
    quality_table,
    caption=(
        "Mittlere relative Lösungsqualität zur Ground-Truth-Partition.  Ein Wert von 1 entspricht derselben Partitionsdichte wie die Ground Truth. Werte kleiner als 1 zeigen eine höhere, Werte größer als 1 eine geringere Partitionsdichte als die Ground Truth."
    ),
    label="tab:ground_truth_quality",
)

print(ground_truth_quality_latex)

\begin{table}[p]
\centering
\caption{Mittlere relative Lösungsqualität zur Ground-Truth-Partition.  Ein Wert von 1 entspricht derselben Partitionsdichte wie die Ground Truth. Werte kleiner als 1 zeigen eine höhere, Werte größer als 1 eine geringere Partitionsdichte als die Ground Truth.}
\label{tab:ground_truth_quality}
\small
\begin{tabular}{p{2.0cm}rlrrrr}
\toprule
Datensatz & $\mu$ & \shortstack{Community-\\größe} & \texttt{Leiden} & \shortstack{\texttt{Leiden-}\\\texttt{MDGP}} & \texttt{KaPoCE} & \shortstack{\textls[-50]{\textsc{MDGP-}}\\\textls[-50]{\textsc{Plateau}}} \\
\midrule
\multirow{4}{*}{small sparse} & \multirow{2}{*}{0.05} & small & 1.0202 & 0.5939 & 0.5534 & \textbf{0.5232} \\
 &  & large & 1.0016 & 0.3244 & 0.2919 & \textbf{0.2836} \\
\cmidrule(l){2-7}
 & \multirow{2}{*}{0.20} & small & 1.0571 & 0.5390 & 0.4759 & \textbf{0.4567} \\
 &  & large & 1.0176 & 0.2870 & 0.2517 & \textbf{0.2469} \\
\midrule
\multirow{4}{*}{small dense} & \multirow{2}{*}{0.05} & small & 1.0177 

In [34]:
def make_ground_truth_f_score_latex_table(
        df: pd.DataFrame,
        caption: str,
        label: str,
) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    noise_order = [0.05, 0.20]
    community_order_by_size = {
        "small": ["small", "large"],
        "large": ["tiny", "small", "large"],
    }

    algorithm_columns = [
        "leiden",
        "leiden_mdgp",
        "kapoce",
        "mdgp_plateau",
    ]

    lines = [
        r"\begin{table}[p]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\small",
        r"\begin{tabular}{p{2.0cm}rlrrrr}",
        r"\toprule",
        r"Datensatz & $\mu$ & \shortstack{Community-\\größe} & \texttt{Leiden} & \shortstack{\texttt{Leiden-}\\\texttt{MDGP}} & \texttt{KaPoCE} & \shortstack{\textls[-50]{\textsc{MDGP-}}\\\textls[-50]{\textsc{Plateau}}} \\",
        r"\midrule",
    ]

    for dataset_index, (size_class, regime) in enumerate(dataset_order):
        dataset_df = df[(df["size_class"] == size_class) & (df["regime"] == regime)].copy()

        community_order = community_order_by_size[size_class]

        dataset_row_count = len(noise_order) * len(community_order)

        current_dataset_row = 0

        for noise_index, noise in enumerate(noise_order):
            noise_df = dataset_df[np.isclose(dataset_df["noise"].astype(float), noise)].copy()

            for community_index, community_size in enumerate(community_order):
                part = noise_df[noise_df["community_size_class"] == community_size]

                row = part.iloc[0]

                values = {algorithm: row[algorithm] for algorithm in algorithm_columns}
                best_f_score = max(values.values())

                formatted_values = {}

                for algorithm, value in values.items():
                    formatted = format_number(value, 4)

                    if np.isclose(value, best_f_score):
                        formatted = rf"\textbf{{{formatted}}}"

                    formatted_values[algorithm] = formatted

                dataset_cell = (
                    rf"\multirow{{{dataset_row_count}}}{{*}}{{{size_class} {regime}}}"
                    if current_dataset_row == 0
                    else ""
                )

                noise_cell = (
                    rf"\multirow{{{len(community_order)}}}{{*}}{{{noise:.2f}}}"
                    if community_index == 0
                    else ""
                )

                lines.append(
                    f"{dataset_cell} "
                    f"& {noise_cell} "
                    f"& {community_size} "
                    f"& {formatted_values['leiden']} "
                    f"& {formatted_values['leiden_mdgp']} "
                    f"& {formatted_values['kapoce']} "
                    f"& {formatted_values['mdgp_plateau']} "
                    r"\\"
                )

                current_dataset_row += 1

            if noise_index < len(noise_order) - 1:
                lines.append(r"\cmidrule(l){2-7}")

        if dataset_index < len(dataset_order) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [35]:
ground_truth_f_score_latex = make_ground_truth_f_score_latex_table(
    f_score_table,
    caption=(
        "Mittlerer Clustering-F-Score der betrachteten Verfahren gegenüber der Ground-Truth-Partition. Höhere Werte zeigen eine stärkere strukturelle Übereinstimmung mit der Ground Truth."
    ),
    label="tab:ground_truth_f_score",
)

print(ground_truth_f_score_latex)

\begin{table}[p]
\centering
\caption{Mittlerer Clustering-F-Score der betrachteten Verfahren gegenüber der Ground-Truth-Partition. Höhere Werte zeigen eine stärkere strukturelle Übereinstimmung mit der Ground Truth.}
\label{tab:ground_truth_f_score}
\small
\begin{tabular}{p{2.0cm}rlrrrr}
\toprule
Datensatz & $\mu$ & \shortstack{Community-\\größe} & \texttt{Leiden} & \shortstack{\texttt{Leiden-}\\\texttt{MDGP}} & \texttt{KaPoCE} & \shortstack{\textls[-50]{\textsc{MDGP-}}\\\textls[-50]{\textsc{Plateau}}} \\
\midrule
\multirow{4}{*}{small sparse} & \multirow{2}{*}{0.05} & small & \textbf{0.9842} & 0.2911 & 0.4371 & 0.3330 \\
 &  & large & \textbf{0.9946} & 0.1534 & 0.2041 & 0.1725 \\
\cmidrule(l){2-7}
 & \multirow{2}{*}{0.20} & small & \textbf{0.9494} & 0.2603 & 0.3838 & 0.3097 \\
 &  & large & \textbf{0.9576} & 0.1438 & 0.1907 & 0.1677 \\
\midrule
\multirow{4}{*}{small dense} & \multirow{2}{*}{0.05} & small & \textbf{0.9860} & 0.4379 & 0.9598 & 0.4629 \\
 &  & large & \textbf{0.9937} & 0